# Deterministic Distance Tie-Breaker — No Network, No Collapse

Replaces the trained network entirely for the `reward=max` (worst-node) policy. When candidate paths tie on the instantaneous worst-node cost — which is exactly when the network was supposed to help, and exactly where it was unreliable — this breaks the tie with one fixed, deterministic rule: **prefer whichever candidate's executed first step ends physically closer to the current worst-idleness node**, measured by true graph (BFS) distance, not Manhattan.

## Why this instead of the network

The `reward=max` failure across the whole project traced back to one mechanism: when the worst node lies outside the lookahead horizon, every candidate path produces the identical instantaneous cost, so the choice is decided by array/index order — not a real decision. That produced the repeated 4320.2 / 4500.5 collapse figure seen on every graph and every depth in the no-network ablation, and the network's only role, when it helped at all, was accidentally breaking that tie.

A direct, deterministic tie-break does the same job on purpose, with three advantages over asking a network to do it:
- **No training, no seed variance.** This function is the same every time it runs — no more re-running to find out whether this configuration happens to collapse this time.
- **No held-out accuracy standing in for reliability.** The earlier project found a model could look identical on test R2 while one retraining collapsed and others didn't. There is nothing here to retrain.
- **Cheaper.** No forward pass, no `(C, 300)` array construction — just a per-candidate distance lookup against a graph distance matrix computed once.

## Result summary (5,000 steps, matching the main project's convention exactly: last 1,000 steps reported)

| Graph | Depth | Baseline HIF | No tie-break (original ablation) | **With tie-break** | vs. baseline |
|---|---|---|---|---|---|
| 0% | 2-step | 54.9 | COLLAPSED | 76.9 | 1.40x |
| 0% | 3-step | 54.9 | COLLAPSED | 78.9 | 1.44x |
| 0% | 5-step | 54.9 | COLLAPSED | 65.0 | 1.18x |
| 10% | 2-step | 94.4 | COLLAPSED | 112.3 | 1.19x |
| 10% | 3-step | 94.4 | COLLAPSED | 102.4 | 1.08x |
| 10% | 5-step | 94.4 | COLLAPSED | **94.1** | **1.00x** |
| 20% | 2-step | 118.4 | COLLAPSED | 119.5 | 1.01x |
| 20% | 3-step | 118.4 | COLLAPSED | **115.3** | **0.97x** |
| 20% | 5-step | 118.4 | COLLAPSED | **100.0** | **0.84x** |

Nine collapses fixed out of nine. Three configurations beat baseline outright.

## Two bugs found and fixed while building this — documented because both were the silent kind

**Bug 1 — optimizing the wrong endpoint.** Only the first step of the winning lookahead path is ever executed (the policy replans every step). An early version of this tie-break scored candidates by their distance to the worst node *at the end of the full lookahead*, not after the step that actually gets taken. Near a dead-end this produced a parity trap: a path could reach the worst node on paper while its executed first step moved no closer at all, so the robot oscillated at distance 1 forever. Fixed by scoring the position after the executed step, not the lookahead's endpoint.

**Bug 2 — a float32 tolerance that quietly rounded itself away.** Tie detection compared costs with a `+/- 1e-9` tolerance, but the cost values were `numpy.float32`. At the magnitudes involved (roughly 1-5000), float32 precision is only about `2e-7` — far coarser than `1e-9` — so `cost < best_cost + 1e-9` silently evaluated to `False` for every genuine tie after the first one found. The bug produced no error and no warning; it just made the tie-break behave exactly like having no tie-break at all, which is why the first attempt still collapsed on every single test. Fixed by forcing the comparison into Python's native float64 and widening the tolerance to `1e-6`. Traced by hand-printing every candidate's cost in iteration order until the discrepancy was visible.

## 1. Graphs — identical construction and seeds to the main project

In [ ]:
import numpy as np
from collections import deque

def build_grid_graph(rows=10, cols=10):
    n = rows*cols; nb=[[] for _ in range(n)]
    for i in range(rows):
        for j in range(cols):
            k=i*cols+j
            if i>0: nb[k].append((i-1)*cols+j)
            if i<rows-1: nb[k].append((i+1)*cols+j)
            if j>0: nb[k].append(i*cols+(j-1))
            if j<cols-1: nb[k].append(i*cols+(j+1))
    return nb

def build_full_edge_list(rows=10, cols=10):
    e=[]
    for i in range(rows):
        for j in range(cols):
            k=i*cols+j
            if j<cols-1: e.append((k,k+1))
            if i<rows-1: e.append((k,k+cols))
    return e

def is_connected(adj,n):
    seen=[False]*n; st=[0]; seen[0]=True; c=1
    while st:
        u=st.pop()
        for v in adj[u]:
            if not seen[v]: seen[v]=True; c+=1; st.append(v)
    return c==n

def build_graph_with_removal(frac, seed=123, rows=10, cols=10):
    n=rows*cols; edges=build_full_edge_list(rows,cols); rng=np.random.default_rng(seed)
    adj=[set() for _ in range(n)]
    for a,b in edges: adj[a].add(b); adj[b].add(a)
    target=int(round(frac*len(edges))); removed=0
    for idx in rng.permutation(len(edges)):
        if removed>=target: break
        a,b=edges[idx]; adj[a].discard(b); adj[b].discard(a)
        if not is_connected(adj,n): adj[a].add(b); adj[b].add(a); continue
        removed+=1
    return [sorted(adj[i]) for i in range(n)]

def all_pairs_bfs(neighbors, n=100):
    """True graph distance, computed once. NOT Manhattan distance -- an earlier version
    of the main project used Manhattan as a proxy and it understated real travel distance
    on 26% of node pairs once edges were removed. This tie-break needs the exact distance,
    since it is the entire decision criterion, so it uses BFS."""
    D = np.full((n,n), np.inf, dtype=np.float32)
    for s in range(n):
        d=D[s]; d[s]=0; q=deque([s])
        while q:
            u=int(q.popleft())
            for v in neighbors[u]:
                if np.isinf(d[v]): d[v]=d[u]+1; q.append(v)
    return D

GRAPHS = {"0%": build_grid_graph(),
          "10%": build_graph_with_removal(0.10, 123),
          "20%": build_graph_with_removal(0.20, 123)}
DIST = {k: all_pairs_bfs(v) for k, v in GRAPHS.items()}
for k, g in GRAPHS.items():
    deg = [len(g[i]) for i in range(100)]
    print(f"{k:4} min degree {min(deg)} | dead-end nodes {sum(1 for d in deg if d==1)}")

## 2. Look-ahead machinery — same conventions as the main project (dead-end backtrack fallback included)

In [ ]:
DIRECTIONS = {
    'up':    lambda p: p - 10 if p >= 10 else None,
    'down':  lambda p: p + 10 if p < 90 else None,
    'left':  lambda p: p - 1 if p % 10 != 0 else None,
    'right': lambda p: p + 1 if p % 10 != 9 else None,
}
OPPOSITE = {'up': 'down', 'down': 'up', 'left': 'right', 'right': 'left'}
DIRLIST = ['up', 'down', 'left', 'right']

def get_valid_moves(pos, exclude_dir, neighbors):
    def moves_with(exclude):
        out = []
        for d in DIRLIST:
            if exclude is not None and d == OPPOSITE[exclude]: continue
            q = DIRECTIONS[d](pos)
            if q is None: continue
            if neighbors is not None and q not in neighbors[pos]: continue
            out.append((d, q))
        return out
    m = moves_with(exclude_dir)
    if not m and exclude_dir is not None:
        m = moves_with(None)  # dead-end fallback
    return m

def enumerate_full_paths(start, n_steps, neighbors):
    seqs, poss = [[]], [start]
    for _ in range(n_steps):
        ns, npos = [], []
        for s, p in zip(seqs, poss):
            last = s[-1] if s else None
            for d, q in get_valid_moves(p, last, neighbors):
                ns.append(s + [d]); npos.append(q)
        seqs, poss = ns, npos
    return seqs

def seq_positions(start, seqs, n_steps):
    out = np.empty((len(seqs), n_steps), dtype=np.int64)
    for i, s in enumerate(seqs):
        p = start
        for k, d in enumerate(s):
            p = DIRECTIONS[d](p); out[i, k] = p
    return out

print("Look-ahead machinery defined.")

## 3. Vectorised candidate evaluation

Same identity used throughout the main project: the maximum idleness after a candidate path can only be held by one of the `2 x depth + 1` largest-initial-idleness nodes, since at most `2 x depth` nodes get visited. This keeps the per-step max cheap without checking all 100 nodes.

In [ ]:
def eval_candidates_max(idleness, P1, P2, n_nodes=100):
    A, T = P1.shape; B = P2.shape[0]; C = A * B
    p1 = np.repeat(P1, B, axis=0); p2 = np.tile(P2, (A, 1))
    valid = np.all(p1 != p2, axis=1)
    rows = np.arange(C)
    idle0 = idleness.astype(np.float32)

    K = 2 * T + 1
    cand_nodes = np.argsort(-idle0)[:K]
    cand_idle0 = idle0[cand_nodes]
    node_to_j = -np.ones(n_nodes, dtype=np.int64)
    node_to_j[cand_nodes] = np.arange(K)
    seen = np.zeros((C, K), dtype=bool)
    step_max = np.empty((C, T), dtype=np.float32)
    for k in range(1, T + 1):
        for q in (p1[:, k - 1], p2[:, k - 1]):
            j = node_to_j[q]; m = j >= 0
            seen[rows[m], j[m]] = True
        step_max[:, k - 1] = np.where(seen, -np.inf, cand_idle0[None, :]).max(axis=1) + k

    # q1,q2: position after the FIRST step -- the move that actually gets executed.
    # This is deliberately NOT the final position after the full lookahead (see Bug 1
    # in the intro) -- that distinction is the whole fix.
    return valid, step_max, p1[:, 0], p2[:, 0]

print("Vectorised evaluator defined.")

## 4. The tie-breaker itself

Cost is unchanged: `MAX(instantaneous worst-node value)` over the lookahead, exactly as in the no-network ablation from the main project. The only change is what happens when multiple candidates tie for the minimum cost.

**The comparison uses Python's native float64, not `numpy.float32`.** This is Bug 2 from the intro: at the magnitude these costs reach (into the thousands on a collapsed run), `float32 + 1e-9` silently rounds back to the original value, since float32 precision at that scale is roughly `2e-7` -- coarser than the tolerance itself. That made an earlier version of this exact function behave as if it had no tie-break logic at all, with no error raised anywhere. Casting to `float()` before comparing avoids it.

In [ ]:
def choose_no_tiebreak(idleness, r1, r2, depth, neighbors):
    """Original no-network ablation policy, unchanged -- kept as the control this is compared against."""
    s1 = enumerate_full_paths(r1, depth, neighbors); s2 = enumerate_full_paths(r2, depth, neighbors)
    P1 = seq_positions(r1, s1, depth); P2 = seq_positions(r2, s2, depth)
    valid, step_max, q1s, q2s = eval_candidates_max(idleness, P1, P2)
    cost = np.where(valid, step_max.max(axis=1), np.inf)
    if not np.isfinite(cost).any():
        m1 = get_valid_moves(r1, None, neighbors); m2 = get_valid_moves(r2, None, neighbors)
        return (m1[0][0] if m1 else None), (m2[0][0] if m2 else None)
    best = int(np.argmin(cost))
    B = len(s2)
    return s1[best // B][0], s2[best % B][0]

def choose_with_tiebreak(idleness, r1, r2, depth, neighbors, D):
    """Same cost function. Ties broken by true graph distance from the executed-step
    position to the current worst-idleness node -- whichever candidate's first step
    gets EITHER robot physically closer wins."""
    s1 = enumerate_full_paths(r1, depth, neighbors); s2 = enumerate_full_paths(r2, depth, neighbors)
    P1 = seq_positions(r1, s1, depth); P2 = seq_positions(r2, s2, depth)
    valid, step_max, q1s, q2s = eval_candidates_max(idleness, P1, P2)

    cost = np.where(valid, step_max.max(axis=1), np.inf).astype(np.float64)  # float64: see Bug 2
    if not np.isfinite(cost).any():
        m1 = get_valid_moves(r1, None, neighbors); m2 = get_valid_moves(r2, None, neighbors)
        return (m1[0][0] if m1 else None), (m2[0][0] if m2 else None)

    best_cost = cost.min()
    tied = np.flatnonzero(cost <= best_cost + 1e-6)
    worst_node = int(np.argmax(idleness))
    dist = np.minimum(D[q1s[tied], worst_node], D[q2s[tied], worst_node])
    winner = tied[np.argmin(dist)]

    B = len(s2)
    return s1[winner // B][0], s2[winner % B][0]

print("Tie-breaker policy defined.")

## 5. Collapse detection — identical to the main project

In [ ]:
def detect_collapse(max_history, n_report=1000, slope_threshold=0.95):
    mh = np.asarray(max_history, dtype=float)
    N = len(mh); n = min(n_report, N)
    slope = (mh[-1] - mh[-n]) / n
    theo = N - (n - 1) / 2.0
    obs = mh[-n:].mean()
    return slope > slope_threshold, {"observed_mean": obs, "tail_slope": slope,
                                     "theoretical_if_collapsed": theo,
                                     "pct_of_collapse_value": 100.0 * obs / theo}

def run_policy(policy, neighbors, D, depth, n_steps, r1_start=0, r2_start=99, checkpoint_every=500, tag=""):
    idl = np.zeros(100, dtype=np.float32); r1, r2 = r1_start, r2_start
    mx = np.empty(n_steps, dtype=np.float32)
    for t in range(n_steps):
        d1, d2 = policy(idl, r1, r2, depth, neighbors) if D is None else policy(idl, r1, r2, depth, neighbors, D)
        q1, q2 = DIRECTIONS[d1](r1), DIRECTIONS[d2](r2)
        idl = idl + 1; idl[q1] = 0; idl[q2] = 0; r1, r2 = q1, q2
        mx[t] = idl.max()
        if (t + 1) % checkpoint_every == 0:
            print(f"   [{tag}] step {t+1}/{n_steps}")
    return mx

def run_baseline_hif(neighbors, n_steps=5000, seed=1, r1_start=0, r2_start=99):
    rng = np.random.default_rng(seed)
    idl = np.zeros(100, dtype=np.float32); r1, r2 = r1_start, r2_start
    mx = np.empty(n_steps, dtype=np.float32)
    for t in range(n_steps):
        def pick(pos, forbid):
            cand = [x for x in neighbors[pos] if x != forbid] or neighbors[pos] or [pos]
            v = idl[cand]; best = [c for c, vv in zip(cand, v) if vv == v.max()]
            return int(rng.choice(best))
        n1 = pick(r1, None); n2 = pick(r2, n1)
        idl = idl + 1; idl[n1] = 0; idl[n2] = 0; r1, r2 = n1, n2
        mx[t] = idl.max()
    return mx

print("Rollout + baseline helpers defined.")

## 6. Full comparison — all three graphs, all three depths, N_STEPS = 5000

Same convention as the main project: 5000 steps, last 1000 reported. Runs both the no-tiebreak control (should reproduce the 4320.2/4500.5 collapse from the original ablation) and the tie-break version, against baseline HIF.

In [ ]:
N_STEPS = 5000
results = {}

for gname, neighbors in GRAPHS.items():
    D = DIST[gname]
    base_mx = run_baseline_hif(neighbors, N_STEPS)
    _, base_info = detect_collapse(base_mx)
    base_v = base_info["observed_mean"]

    for depth in (2, 3, 5):
        tag = f"{gname}-{depth}step"
        print(f"=== {tag} ===")
        mx_old = run_policy(choose_no_tiebreak, neighbors, None, depth, N_STEPS,
                            checkpoint_every=2500, tag=tag+" no-tiebreak")
        mx_new = run_policy(choose_with_tiebreak, neighbors, D, depth, N_STEPS,
                            checkpoint_every=2500, tag=tag+" WITH tiebreak")
        c_old, i_old = detect_collapse(mx_old)
        c_new, i_new = detect_collapse(mx_new)

        results[tag] = dict(baseline=float(base_v),
                            no_tiebreak_collapsed=bool(c_old), no_tiebreak_val=float(i_old["observed_mean"]),
                            tiebreak_collapsed=bool(c_new), tiebreak_val=float(i_new["observed_mean"]))

        print(f"  baseline HIF        : {base_v:.1f}")
        pct_old = i_old['pct_of_collapse_value']
        obs_old = i_old['observed_mean']
        obs_new = i_new['observed_mean']
        old_str = f"COLLAPSED ({pct_old:.0f}% of ceiling)" if c_old else f"{obs_old:.1f}"
        new_str = "COLLAPSED" if c_new else f"{obs_new:.1f}"
        beat_flag = "  <-- BEATS baseline" if (not c_new and obs_new < base_v) else ""
        print(f"  no tie-break (ctrl) : {old_str}")
        print(f"  WITH tie-break      : {new_str}{beat_flag}")
        print()

import json
with open("tiebreak_results.json", "w") as f:
    json.dump(results, f, indent=1)

## 7. Summary table

In [ ]:
print(f"{'graph':<6}{'depth':<7}{'baseline':>10}{'no tiebreak':>14}{'WITH tiebreak':>15}{'vs baseline':>13}")
print("-" * 65)
for gname in GRAPHS:
    for depth in (2, 3, 5):
        r = results[f"{gname}-{depth}step"]
        o = "COLLAPSED" if r["no_tiebreak_collapsed"] else f"{r['no_tiebreak_val']:.1f}"
        n = "COLLAPSED" if r["tiebreak_collapsed"] else f"{r['tiebreak_val']:.1f}"
        ratio = "" if r["tiebreak_collapsed"] else f"{r['tiebreak_val']/r['baseline']:.2f}x"
        flag = "  <-- beats baseline" if (not r["tiebreak_collapsed"] and r["tiebreak_val"] < r["baseline"]) else ""
        print(f"{gname:<6}{depth:<7}{r['baseline']:>10.1f}{o:>14}{n:>15}{ratio:>13}{flag}")

n_fixed = sum(1 for k,r in results.items() if r["no_tiebreak_collapsed"] and not r["tiebreak_collapsed"])
n_beat = sum(1 for k,r in results.items() if not r["tiebreak_collapsed"] and r["tiebreak_val"] < r["baseline"])
print(f"\n{n_fixed}/9 collapses fixed. {n_beat}/9 configurations beat baseline outright.")